# Baseline 06 — TrOCR Fine-Tuning

Pretrained Transformer OCR baseline using Hugging Face TrOCR. It fine-tunes TrOCR and evaluates the same CER and WER metrics. Internet access or a cached `microsoft/trocr-base-stage1` model is required on the server.

In [ ]:
%pip install pandas pillow tqdm matplotlib openpyxl transformers accelerate sentencepiece

In [ ]:
import os
import json
import math
import time
import random
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# Project and processed data
# =========================
cwd = Path.cwd().resolve()
PROJECT_DIR = cwd
if not (PROJECT_DIR / "data").exists():
    for parent in [cwd] + list(cwd.parents):
        if (parent / "data").exists():
            PROJECT_DIR = parent
            break

DATA_DIR = PROJECT_DIR / "data"
assert DATA_DIR.exists(), f"Could not find data directory. Current path: {cwd}"

# Optional: set this manually if you want a specific split folder.
# Example:
# PREFERRED_SPLIT_DIR = DATA_DIR / "processed_line_full" / "splits"
PREFERRED_SPLIT_DIR = None

def load_jsonl_robust(path):
    path = Path(path)
    rows = []
    decoder = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    idx = 0
    n = len(content)
    while idx < n:
        while idx < n and content[idx].isspace():
            idx += 1
        if idx >= n:
            break
        obj, end = decoder.raw_decode(content[idx:])
        rows.append(obj)
        idx += end
    return rows

def count_jsonl(path):
    try:
        return len(load_jsonl_robust(path))
    except Exception:
        return 0

def find_split_dir():
    if PREFERRED_SPLIT_DIR is not None:
        p = Path(PREFERRED_SPLIT_DIR)
        assert (p / "line_train.jsonl").exists(), f"Missing train split: {p}"
        return p

    candidates = []
    for train_path in DATA_DIR.rglob("line_train.jsonl"):
        split_dir = train_path.parent
        val_path = split_dir / "line_val.jsonl"
        test_path = split_dir / "line_test.jsonl"
        if not val_path.exists() or not test_path.exists():
            continue

        proc_dir = split_dir.parent if split_dir.name == "splits" else split_dir
        n_train = count_jsonl(train_path)
        n_val = count_jsonl(val_path)
        n_test = count_jsonl(test_path)

        score = n_train + n_val + n_test
        name = str(proc_dir).lower()
        if "processed_line_full" in name:
            score += 1_000_000
        if "processed_psocr" in name:
            score += 900_000
        if "full" in name:
            score += 100_000
        if (proc_dir / "vocab.json").exists():
            score += 10_000

        candidates.append((score, split_dir, proc_dir, n_train, n_val, n_test))

    assert len(candidates) > 0, "No processed line-level split files found under data/."
    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    print("Detected split candidates:")
    for score, split_dir, proc_dir, n_train, n_val, n_test in candidates[:10]:
        print(f"score={score} | train/val/test={n_train}/{n_val}/{n_test} | {split_dir}")

    return candidates[0][1]

SPLIT_DIR = find_split_dir()
PROC_DIR = SPLIT_DIR.parent if SPLIT_DIR.name == "splits" else SPLIT_DIR

paths = {
    "train": SPLIT_DIR / "line_train.jsonl",
    "val": SPLIT_DIR / "line_val.jsonl",
    "test": SPLIT_DIR / "line_test.jsonl",
}

print("\nSelected SPLIT_DIR:", SPLIT_DIR)
print("Selected PROC_DIR:", PROC_DIR)
for k, p in paths.items():
    print(k, count_jsonl(p), p)

# =========================
# Vocabulary
# =========================
def normalize_pashto_text(text):
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200f", "")
    text = text.replace("\ufeff", "")
    text = " ".join(text.split())
    return text.strip()

vocab_path = PROC_DIR / "vocab.json"
if vocab_path.exists():
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)
    print("Loaded vocab:", vocab_path)
else:
    print("vocab.json not found. Building vocabulary from train split only.")
    train_records_for_vocab = load_jsonl_robust(paths["train"])
    chars = sorted(set("".join(normalize_pashto_text(r["text"]) for r in train_records_for_vocab)))
    itos_tmp = ["<blank>", "<unk>"] + chars
    stoi_tmp = {ch: i for i, ch in enumerate(itos_tmp)}
    vocab = {"itos": itos_tmp, "stoi": stoi_tmp, "blank_id": 0, "unk_id": 1}
    vocab_path = PROC_DIR / "vocab_train_only_auto.json"
    with open(vocab_path, "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False, indent=2)
    print("Saved auto vocab:", vocab_path)

itos = vocab["itos"]
stoi = vocab["stoi"]
BLANK_ID = vocab["blank_id"]
UNK_ID = vocab["unk_id"]

print("Vocab size:", len(itos))

# =========================
# Image resolver
# =========================
image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}
image_map = {}
for p in DATA_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in image_exts:
        image_map[p.name] = p

def resolve_image_path(path_value):
    p = Path(str(path_value))
    if p.exists():
        return p
    for base in [PROJECT_DIR, DATA_DIR, PROC_DIR]:
        q = base / p
        if q.exists():
            return q
    if p.name in image_map:
        return image_map[p.name]
    raise FileNotFoundError(f"Image not found: {path_value}")

# =========================
# Dataset
# =========================
def encode_text(text):
    return [stoi.get(ch, UNK_ID) for ch in normalize_pashto_text(text)]

def decode_ids(ids):
    chars = []
    for i in ids:
        i = int(i)
        if i == BLANK_ID:
            continue
        if i < 0 or i >= len(itos):
            continue
        ch = itos[i]
        if ch in ["<blank>", "<unk>"]:
            continue
        chars.append(ch)
    return "".join(chars)

def preprocess_image(path, img_height=48, max_width=1024):
    img_path = resolve_image_path(path)
    img = Image.open(img_path).convert("L")
    arr0 = np.array(img)
    if arr0.mean() < 127:
        img = ImageOps.invert(img)

    w, h = img.size
    new_w = max(16, int(round(w * img_height / max(1, h))))
    new_w = min(new_w, max_width)

    resample = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
    img = img.resize((new_w, img_height), resample)

    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0), new_w

class PashtoLineOCRDataset(Dataset):
    def __init__(self, jsonl_path, img_height=48, max_width=1024):
        self.records = load_jsonl_robust(jsonl_path)
        self.records = [r for r in self.records if normalize_pashto_text(r.get("text", ""))]
        self.img_height = img_height
        self.max_width = max_width

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image, width = preprocess_image(rec["image"], self.img_height, self.max_width)
        text = normalize_pashto_text(rec["text"])
        label = torch.tensor(encode_text(text), dtype=torch.long)
        return {"image": image, "width": width, "label": label, "text": text, "path": rec["image"]}

def collate_ocr(batch):
    H = max(b["image"].shape[1] for b in batch)
    W = max(b["image"].shape[2] for b in batch)
    images = torch.full((len(batch), 1, H, W), 1.0, dtype=torch.float32)
    for i, b in enumerate(batch):
        img = b["image"]
        _, h, w = img.shape
        images[i, :, :h, :w] = img

    image_widths = torch.tensor([b["width"] for b in batch], dtype=torch.long)
    labels = [b["label"] for b in batch]
    target_lengths = torch.tensor([len(x) for x in labels], dtype=torch.long)
    targets = torch.cat(labels, dim=0)
    return {
        "images": images,
        "image_widths": image_widths,
        "targets": targets,
        "target_lengths": target_lengths,
        "texts": [b["text"] for b in batch],
        "paths": [b["path"] for b in batch],
    }

# =========================
# Metrics
# =========================
def greedy_decode_ctc(logits, input_lengths):
    pred_ids = logits.argmax(dim=-1).detach().cpu().numpy()
    input_lengths = input_lengths.detach().cpu().numpy()
    texts = []
    for b in range(pred_ids.shape[0]):
        seq = pred_ids[b, :input_lengths[b]]
        collapsed = []
        prev = None
        for idx in seq:
            idx = int(idx)
            if idx != BLANK_ID and idx != prev:
                collapsed.append(idx)
            prev = idx
        texts.append(decode_ids(collapsed))
    return texts

def edit_distance(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    curr = [0] * (m + 1)
    for i in range(1, n + 1):
        curr[0] = i
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev, curr = curr, prev
    return prev[m]

def cer(pred, gold):
    pred = normalize_pashto_text(pred)
    gold = normalize_pashto_text(gold)
    if len(gold) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return edit_distance(pred, gold) / len(gold)

def wer(pred, gold):
    pred_words = normalize_pashto_text(pred).split()
    gold_words = normalize_pashto_text(gold).split()
    if len(gold_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return edit_distance(pred_words, gold_words) / len(gold_words)

def format_seconds(seconds):
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

print("Common utilities loaded for TrOCR.")

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_NAME = "microsoft/trocr-base-stage1"

class TrOCRLineDataset(Dataset):
    def __init__(self, jsonl_path, processor, max_target_length=160):
        self.records = load_jsonl_robust(jsonl_path)
        self.records = [r for r in self.records if normalize_pashto_text(r.get("text", ""))]
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = Image.open(resolve_image_path(rec["image"])).convert("RGB")
        text = normalize_pashto_text(rec["text"])
        pixel_values = self.processor(images=img, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels, "text": text, "path": rec["image"]}

def collate_trocr(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
        "texts": [b["text"] for b in batch],
        "paths": [b["path"] for b in batch],
    }

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASELINE_NAME = "trocr_finetune"
OUTPUT_DIR = PROJECT_DIR / "outputs" / f"baseline_{BASELINE_NAME}"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
for p in [OUTPUT_DIR, CHECKPOINT_DIR, PREDICTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 4
EPOCHS = 50
LR = 5e-5
WEIGHT_DECAY = 1e-4
MAX_TARGET_LENGTH = 160
NUM_WORKERS = 0

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Important decoder settings
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = MAX_TARGET_LENGTH
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 0
model.config.length_penalty = 1.0
model.config.num_beams = 1

model.to(DEVICE)

train_ds = TrOCRLineDataset(paths["train"], processor, MAX_TARGET_LENGTH)
val_ds = TrOCRLineDataset(paths["val"], processor, MAX_TARGET_LENGTH)
test_ds = TrOCRLineDataset(paths["test"], processor, MAX_TARGET_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_trocr)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_trocr)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_trocr)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))

history = {"train_loss": [], "val_cer": [], "val_wer": [], "epoch_time_sec": []}
epoch_rows = []
best_val_cer = float("inf")
excel_path = OUTPUT_DIR / "trocr_finetune_training_time.xlsx"
best_dir = CHECKPOINT_DIR / "best_trocr"
last_dir = CHECKPOINT_DIR / "last_trocr"

def save_excel(epoch_rows, summary_rows):
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        pd.DataFrame(epoch_rows).to_excel(writer, index=False, sheet_name="Epoch_Timing")
        pd.DataFrame(summary_rows).to_excel(writer, index=False, sheet_name="Summary")

def generate_predictions(loader):
    model.eval()
    all_pred, all_gold, all_paths = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Generating", leave=False):
            pixel_values = batch["pixel_values"].to(DEVICE)
            generated_ids = model.generate(pixel_values, max_length=MAX_TARGET_LENGTH, num_beams=1)
            preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
            all_pred.extend([normalize_pashto_text(p) for p in preds])
            all_gold.extend(batch["texts"])
            all_paths.extend(batch["paths"])
    return all_pred, all_gold, all_paths

def run_eval(loader):
    preds, golds, _ = generate_predictions(loader)
    mean_cer = float(np.mean([cer(p, g) for p, g in zip(preds, golds)])) if golds else 1.0
    mean_wer = float(np.mean([wer(p, g) for p, g in zip(preds, golds)])) if golds else 1.0
    return mean_cer, mean_wer

start = time.perf_counter()
start_clock = time.strftime("%Y-%m-%d %H:%M:%S")

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    model.train()
    total_loss = 0.0
    batches = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        batches += 1

    scheduler.step()
    train_loss = total_loss / max(1, batches)
    val_cer, val_wer = run_eval(val_loader)

    epoch_time = time.perf_counter() - epoch_start
    elapsed = time.perf_counter() - start
    remaining = (elapsed / epoch) * (EPOCHS - epoch)

    history["train_loss"].append(train_loss)
    history["val_cer"].append(val_cer)
    history["val_wer"].append(val_wer)
    history["epoch_time_sec"].append(epoch_time)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_CER": val_cer,
        "val_WER": val_wer,
        "epoch_time_seconds": epoch_time,
        "epoch_time_hms": format_seconds(epoch_time),
        "elapsed_hms": format_seconds(elapsed),
        "estimated_remaining_hms": format_seconds(remaining),
        "learning_rate": optimizer.param_groups[0]["lr"],
    }
    epoch_rows.append(row)

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_CER={val_cer:.4f}, val_WER={val_wer:.4f}, epoch_time={format_seconds(epoch_time)}")

    model.save_pretrained(last_dir)
    processor.save_pretrained(last_dir)

    if np.isfinite(val_cer) and val_cer < best_val_cer:
        best_val_cer = val_cer
        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
        print("Saved best:", best_dir)

    save_excel(epoch_rows, [{"status": "running", "baseline": BASELINE_NAME, "last_epoch": epoch, "best_val_CER": best_val_cer}])

total = time.perf_counter() - start
save_excel(epoch_rows, [{
    "status": "finished",
    "baseline": BASELINE_NAME,
    "start_time": start_clock,
    "end_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "total_training_time_hms": format_seconds(total),
    "best_val_CER": best_val_cer,
}])

# Load best and test
model = VisionEncoderDecoderModel.from_pretrained(best_dir).to(DEVICE)
processor = TrOCRProcessor.from_pretrained(best_dir)

preds, golds, path_list = generate_predictions(test_loader)
test_cer = float(np.mean([cer(p, g) for p, g in zip(preds, golds)]))
test_wer = float(np.mean([wer(p, g) for p, g in zip(preds, golds)]))
print("Test CER:", test_cer)
print("Test WER:", test_wer)

prediction_rows = []
for path_, gold, pred in zip(path_list, golds, preds):
    prediction_rows.append({"image": path_, "gold": gold, "prediction": pred, "cer": cer(pred, gold), "wer": wer(pred, gold)})

pred_csv = PREDICTION_DIR / "trocr_finetune_test_predictions.csv"
pd.DataFrame(prediction_rows).to_csv(pred_csv, index=False, encoding="utf-8-sig")
print("Saved predictions:", pred_csv)
print("Saved Excel:", excel_path)